In [1]:
%%bash
git clone https://github.com/phamson02/libsmctrl.git
cd libsmctrl
git checkout main
make libsmctrl.a

Your branch is up to date with 'origin/main'.
make: 'libsmctrl.a' is up to date.


fatal: destination path 'libsmctrl' already exists and is not an empty directory.
Already on 'main'


- BLOCK_DIM = (32, 32) -> 1024 threads -> 32 warps -> 8 SMs -> 4 TPCs
- GRID = (120, 120)
- A, B, C = 3840x3840






In [ ]:
# @title
%%writefile gemm_naive.cu
#include <iostream>
#include <tuple>
#include <cassert>

using namespace std;

template<typename T>
__global__ void matmul_kernel(T const* a, T const* b, T* c, int M, int N, int K) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (col < N && row < M) {
        T sum = 0;
        for (int k = 0; k < K; k++) {
            sum += a[row * K + k] * b[k * N + col];
        }
        c[row * N + col] = sum;
    }
}

template<typename T>
__host__ void verifyResult(T* a, T* b, T* c, int M, int N, int K) {
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < N; j++) {
            T sum = 0;
            for (int k = 0; k < K; k++) {
                sum += a[i * K + k] * b[k * N + j];
            }
            assert(c[i * N + j] == sum);
        }
    }
    cout << "Result is correct";
}

template<typename T>
__host__ void copyFromHostToDevice(T* h_a, T* h_b, T* d_a, T* d_b, int M, int N, int K) {
    size_t a_bytes = M * K * sizeof(T);
    size_t b_bytes = K * N * sizeof(T);
    cudaError_t err = cudaMemcpy(d_a, h_a, a_bytes, cudaMemcpyHostToDevice);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy h_a to d_a: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaMemcpy(d_b, h_b, b_bytes, cudaMemcpyHostToDevice);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy h_b to d_b: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template<typename T>
__host__ void executeKernel(T* d_a, T* d_b, T* d_c, int M, int N, int K) {
    int BLOCK_DIM = 32;
    dim3 block(BLOCK_DIM, BLOCK_DIM, 1);
    dim3 grid((N + BLOCK_DIM - 1) / BLOCK_DIM, (M + BLOCK_DIM - 1) / BLOCK_DIM, 1);
    matmul_kernel<T><<<grid, block>>>(d_a, d_b, d_c, M, N, K);
    cudaDeviceSynchronize();

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to launch kernel: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template <typename KernelFunc>
void benchmarkKernel(KernelFunc kernelFunc, int numTrials) {
    float* timings = new float[numTrials];

    // Create CUDA events for timing.
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Run the provided kernel launcher function multiple times.
    for (int i = 0; i < numTrials; i++) {
        cudaEventRecord(start, 0);
        kernelFunc();  // Launch the kernel via the callable.
        cudaEventRecord(stop, 0);
        cudaEventSynchronize(stop);

        float ms;
        cudaEventElapsedTime(&ms, start, stop);
        timings[i] = ms;
    }

    // Clean up CUDA events.
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    // Compute the average execution time.
    float sum = 0.0f;
    for (int i = 0; i < numTrials; i++) {
        sum += timings[i];
    }
    float average = sum / numTrials;

    // Compute the standard deviation.
    float variance = 0.0f;
    for (int i = 0; i < numTrials; i++) {
        float diff = timings[i] - average;
        variance += diff * diff;
    }
    variance /= numTrials;
    float stddev = sqrt(variance);

    // Print the results.
    printf("-----NUM TRIALS: %d -----\n", numTrials);
    printf("Average execution time: %f ms\n", average);
    printf("Standard deviation: %f ms\n", stddev);

    delete[] timings;
}

template<typename T>
__host__ void copyFromDeviceToHost(T* d_c, T* h_c, int M, int N) {
    size_t bytes = M * N * sizeof(T);
    cudaError_t err = cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy d_c to h_c: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template<typename T>
__host__ void deallocateMemory(T* d_a, T* d_b, T* d_c) {
    cudaError_t err = cudaFree(d_a);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_a: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaFree(d_b);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_b: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaFree(d_c);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_c: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

__host__ void cleanUpDevice() {
    cudaError_t err = cudaDeviceReset();
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to clean up device: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

__host__ std::tuple<int, int, int, int, int> parseCmdLineArgs(int argc, char *argv[]) {
    int M = 1024;
    int N = 1024;
    int K = 1024;
    int num_trials = 10;
    int num_warmups = 10;

    for (int i = 1; i < argc; i++) {
        std::string option(argv[i]);
        if (i + 1 < argc) {
            std::string value(argv[i + 1]);
            if (option == "-m") {
                M = std::stoi(value);
                i++;
            } else if (option == "-n") {
                N = std::stoi(value);
                i++;
            } else if (option == "-k") {
                K = std::stoi(value);
                i++;
            } else if (option == "--num_trials") {
                num_trials = std::stoi(value);
                i++;
            } else if (option == "--num_warmups") {
                num_warmups = std::stoi(value);
                i++;
            }
        } else {
            std::cerr << "Missing value for option " << option << std::endl;
        }
    }

    return std::make_tuple(M, N, K, num_trials, num_warmups);
}

int main(int argc, char *argv[]) {
    std::tuple<int, int, int, int, int> parsedCmdLineArgsTuple = parseCmdLineArgs(argc, argv);
    int M = std::get<0>(parsedCmdLineArgsTuple);
    int N = std::get<1>(parsedCmdLineArgsTuple);
    int K = std::get<2>(parsedCmdLineArgsTuple);
    int numTrials = std::get<3>(parsedCmdLineArgsTuple);
    int nWarmUps = std::get<4>(parsedCmdLineArgsTuple);

    int* h_a = (int*)malloc(M * K * sizeof(int));
    int* h_b = (int*)malloc(K * N * sizeof(int));
    int* h_c = (int*)malloc(M * N * sizeof(int));

    srand(42);

    for (size_t i = 0; i < M; i++) {
        for (size_t j = 0; j < K; j++) {
            h_a[i * K + j] = rand() % 10;
        }
    }

    for (size_t i = 0; i < K; i++) {
        for (size_t j = 0; j < N; j++) {
            h_b[i * N + j] = rand() % 10;
        }
    }

    int *d_a, *d_b, *d_c;
    cudaMalloc((int**)&d_a, M * K * sizeof(int));
    cudaMalloc((int**)&d_b, K * N * sizeof(int));
    cudaMalloc((int**)&d_c, M * N * sizeof(int));

    copyFromHostToDevice<int>(h_a, h_b, d_a, d_b, M, N, K);

    for (int i = 0; i < nWarmUps; i++) {
        executeKernel<int>(d_a, d_b, d_c, M, N, K);
    }

    auto kernelLauncher = [&]() {
        executeKernel<int>(d_a, d_b, d_c, M, N, K);
    };

    benchmarkKernel(kernelLauncher, numTrials);

    copyFromDeviceToHost<int>(d_c, h_c, M, N);
    //verifyResult<int>(h_a, h_b, h_c, M, N, K);
    deallocateMemory<int>(d_a, d_b, d_c);
    cleanUpDevice();

    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}

Overwriting gemm_naive.cu


In [ ]:
# @title
%%writefile gemm_naive_with_tpc_masking.cu
#include <iostream>
#include <tuple>
#include <cassert>
#include "libsmctrl/libsmctrl.h"

using namespace std;

template<typename T>
__global__ void matmul_kernel(T const* a, T const* b, T* c, int M, int N, int K) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (col < N && row < M) {
        T sum = 0;
        for (int k = 0; k < K; k++) {
            sum += a[row * K + k] * b[k * N + col];
        }
        c[row * N + col] = sum;
    }
}

template<typename T>
__host__ void verifyResult(T* a, T* b, T* c, int M, int N, int K) {
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < N; j++) {
            T sum = 0;
            for (int k = 0; k < K; k++) {
                sum += a[i * K + k] * b[k * N + j];
            }
            assert(c[i * N + j] == sum);
        }
    }
    cout << "Result is correct";
}

template<typename T>
__host__ void copyFromHostToDevice(T* h_a, T* h_b, T* d_a, T* d_b, int M, int N, int K) {
    size_t a_bytes = M * K * sizeof(T);
    size_t b_bytes = K * N * sizeof(T);
    cudaError_t err = cudaMemcpy(d_a, h_a, a_bytes, cudaMemcpyHostToDevice);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy h_a to d_a: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaMemcpy(d_b, h_b, b_bytes, cudaMemcpyHostToDevice);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy h_b to d_b: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template<typename T>
__host__ void executeKernel(T* d_a, T* d_b, T* d_c, int M, int N, int K) {
    libsmctrl_set_next_mask(0x0);

    int BLOCK_DIM = 32;
    dim3 block(BLOCK_DIM, BLOCK_DIM, 1);
    dim3 grid((N + BLOCK_DIM - 1) / BLOCK_DIM, (M + BLOCK_DIM - 1) / BLOCK_DIM, 1);
    matmul_kernel<T><<<grid, block>>>(d_a, d_b, d_c, M, N, K);
    cudaDeviceSynchronize();

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to launch kernel: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template <typename KernelFunc>
void benchmarkKernel(KernelFunc kernelFunc, int numTrials) {
    float* timings = new float[numTrials];

    // Create CUDA events for timing.
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Run the provided kernel launcher function multiple times.
    for (int i = 0; i < numTrials; i++) {
        cudaEventRecord(start, 0);
        kernelFunc();  // Launch the kernel via the callable.
        cudaEventRecord(stop, 0);
        cudaEventSynchronize(stop);

        float ms;
        cudaEventElapsedTime(&ms, start, stop);
        timings[i] = ms;
    }

    // Clean up CUDA events.
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    // Compute the average execution time.
    float sum = 0.0f;
    for (int i = 0; i < numTrials; i++) {
        sum += timings[i];
    }
    float average = sum / numTrials;

    // Compute the standard deviation.
    float variance = 0.0f;
    for (int i = 0; i < numTrials; i++) {
        float diff = timings[i] - average;
        variance += diff * diff;
    }
    variance /= numTrials;
    float stddev = sqrt(variance);

    // Print the results.
    printf("-----NUM TRIALS: %d -----\n", numTrials);
    printf("Average execution time: %f ms\n", average);
    printf("Standard deviation: %f ms\n", stddev);

    delete[] timings;
}

template<typename T>
__host__ void copyFromDeviceToHost(T* d_c, T* h_c, int M, int N) {
    size_t bytes = M * N * sizeof(T);
    cudaError_t err = cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy d_c to h_c: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template<typename T>
__host__ void deallocateMemory(T* d_a, T* d_b, T* d_c) {
    cudaError_t err = cudaFree(d_a);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_a: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaFree(d_b);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_b: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaFree(d_c);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_c: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

__host__ void cleanUpDevice() {
    cudaError_t err = cudaDeviceReset();
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to clean up device: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

__host__ std::tuple<int, int, int, int, int> parseCmdLineArgs(int argc, char *argv[]) {
    int M = 1024;
    int N = 1024;
    int K = 1024;
    int num_trials = 10;
    int num_warmups = 10;

    for (int i = 1; i < argc; i++) {
        std::string option(argv[i]);
        if (i + 1 < argc) {
            std::string value(argv[i + 1]);
            if (option == "-m") {
                M = std::stoi(value);
                i++;
            } else if (option == "-n") {
                N = std::stoi(value);
                i++;
            } else if (option == "-k") {
                K = std::stoi(value);
                i++;
            } else if (option == "--num_trials") {
                num_trials = std::stoi(value);
                i++;
            } else if (option == "--num_warmups") {
                num_warmups = std::stoi(value);
                i++;
            }
        } else {
            std::cerr << "Missing value for option " << option << std::endl;
        }
    }

    return std::make_tuple(M, N, K, num_trials, num_warmups);
}

int main(int argc, char *argv[]) {
    std::tuple<int, int, int, int, int> parsedCmdLineArgsTuple = parseCmdLineArgs(argc, argv);
    int M = std::get<0>(parsedCmdLineArgsTuple);
    int N = std::get<1>(parsedCmdLineArgsTuple);
    int K = std::get<2>(parsedCmdLineArgsTuple);
    int numTrials = std::get<3>(parsedCmdLineArgsTuple);
    int nWarmUps = std::get<4>(parsedCmdLineArgsTuple);

    int* h_a = (int*)malloc(M * K * sizeof(int));
    int* h_b = (int*)malloc(K * N * sizeof(int));
    int* h_c = (int*)malloc(M * N * sizeof(int));

    srand(42);

    for (size_t i = 0; i < M; i++) {
        for (size_t j = 0; j < K; j++) {
            h_a[i * K + j] = rand() % 10;
        }
    }

    for (size_t i = 0; i < K; i++) {
        for (size_t j = 0; j < N; j++) {
            h_b[i * N + j] = rand() % 10;
        }
    }

    int *d_a, *d_b, *d_c;
    cudaMalloc((int**)&d_a, M * K * sizeof(int));
    cudaMalloc((int**)&d_b, K * N * sizeof(int));
    cudaMalloc((int**)&d_c, M * N * sizeof(int));

    copyFromHostToDevice<int>(h_a, h_b, d_a, d_b, M, N, K);

    for (int i = 0; i < nWarmUps; i++) {
        executeKernel<int>(d_a, d_b, d_c, M, N, K);
    }

    auto kernelLauncher = [&]() {
        executeKernel<int>(d_a, d_b, d_c, M, N, K);
    };

    benchmarkKernel(kernelLauncher, numTrials);

    copyFromDeviceToHost<int>(d_c, h_c, M, N);
    //verifyResult<int>(h_a, h_b, h_c, M, N, K);
    deallocateMemory<int>(d_a, d_b, d_c);
    cleanUpDevice();

    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}

Overwriting gemm_naive_with_tpc_masking.cu


In [4]:
%%bash
nvcc -I./libsmctrl -L./libsmctrl -lsmctrl -lcuda -o gemm_naive_with_tpc_masking gemm_naive_with_tpc_masking.cu -arch=sm_75
./gemm_naive_with_tpc_masking -m 3840 -n 3840 -k 3840 --num_trials 1000 --num_warmups 100

-----NUM TRIALS: 1000 -----
Average execution time: 180.051514 ms
Standard deviation: 3.273903 ms


In [5]:
!nvcc -o gemm_naive gemm_naive.cu -arch=sm_75
!./gemm_naive -m 3840 -n 3840 -k 3840 --num_trials 1000 --num_warmups 100

-----NUM TRIALS: 1000 -----
Average execution time: 187.973083 ms
Standard deviation: 1.114917 ms


In [ ]:
# @title
%%writefile gemm_naive_2stream.cu
#include <iostream>
#include <tuple>
#include <cassert>

using namespace std;

template<typename T>
__global__ void matmul_kernel(T const* a, T const* b, T* c, int M, int N, int K, int row_offset) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    int global_row = row + row_offset;

    if (col < N && row < M) {
        T sum = 0;
        for (int k = 0; k < K; k++) {
            sum += a[global_row * K + k] * b[k * N + col];
        }
        c[global_row * N + col] = sum;
    }
}

template<typename T>
__host__ void verifyResult(T* a, T* b, T* c, int M, int N, int K) {
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < N; j++) {
            T sum = 0;
            for (int k = 0; k < K; k++) {
                sum += a[i * K + k] * b[k * N + j];
            }
            assert(c[i * N + j] == sum);
        }
    }
    cout << "Result is correct";
}

template<typename T>
__host__ void copyFromHostToDevice(T* h_a, T* h_b, T* d_a, T* d_b, int M, int N, int K) {
    size_t a_bytes = M * K * sizeof(T);
    size_t b_bytes = K * N * sizeof(T);
    cudaError_t err = cudaMemcpy(d_a, h_a, a_bytes, cudaMemcpyHostToDevice);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy h_a to d_a: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaMemcpy(d_b, h_b, b_bytes, cudaMemcpyHostToDevice);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy h_b to d_b: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template<typename T>
__host__ void executeKernel(T* d_a, T* d_b, T* d_c, int M, int N, int K) {
    int nStreams = 2;

    cudaStream_t stream[nStreams];
    for (int i = 0; i < nStreams; i ++) {
        cudaStreamCreate(&stream[i]);
    }

    int BLOCK_DIM = 32;
    dim3 block(BLOCK_DIM, BLOCK_DIM, 1);

    int M1 = M / 2;          // first half
    int M2 = M - M1;         // remaining rows

    dim3 grid1((N + BLOCK_DIM - 1) / BLOCK_DIM, (M1 + BLOCK_DIM - 1) / BLOCK_DIM, 1);
    dim3 grid2((N + BLOCK_DIM - 1) / BLOCK_DIM, (M2 + BLOCK_DIM - 1) / BLOCK_DIM, 1);

    matmul_kernel<T><<<grid1, block, 0, stream[0]>>>(d_a, d_b, d_c, M1, N, K, 0);
    matmul_kernel<T><<<grid2, block, 0, stream[1]>>>(d_a, d_b, d_c, M2, N, K, M1);

    for (int i = 0; i < nStreams; i ++) {
        cudaStreamSynchronize(stream[i]);
    }

    for (int i = 0; i < nStreams; i ++) {
        cudaStreamDestroy(stream[i]);
    }

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to launch kernel: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template <typename KernelFunc>
void benchmarkKernel(KernelFunc kernelFunc, int numTrials) {
    float* timings = new float[numTrials];

    // Create CUDA events for timing.
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Run the provided kernel launcher function multiple times.
    for (int i = 0; i < numTrials; i++) {
        cudaEventRecord(start, 0);
        kernelFunc();  // Launch the kernel via the callable.
        cudaEventRecord(stop, 0);
        cudaEventSynchronize(stop);

        float ms;
        cudaEventElapsedTime(&ms, start, stop);
        timings[i] = ms;
    }

    // Clean up CUDA events.
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    // Compute the average execution time.
    float sum = 0.0f;
    for (int i = 0; i < numTrials; i++) {
        sum += timings[i];
    }
    float average = sum / numTrials;

    // Compute the standard deviation.
    float variance = 0.0f;
    for (int i = 0; i < numTrials; i++) {
        float diff = timings[i] - average;
        variance += diff * diff;
    }
    variance /= numTrials;
    float stddev = sqrt(variance);

    // Print the results.
    printf("-----NUM TRIALS: %d -----\n", numTrials);
    printf("Average execution time: %f ms\n", average);
    printf("Standard deviation: %f ms\n", stddev);

    delete[] timings;
}

template<typename T>
__host__ void copyFromDeviceToHost(T* d_c, T* h_c, int M, int N) {
    size_t bytes = M * N * sizeof(T);
    cudaError_t err = cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy d_c to h_c: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template<typename T>
__host__ void deallocateMemory(T* d_a, T* d_b, T* d_c) {
    cudaError_t err = cudaFree(d_a);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_a: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaFree(d_b);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_b: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaFree(d_c);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_c: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

__host__ void cleanUpDevice() {
    cudaError_t err = cudaDeviceReset();
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to clean up device: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

__host__ std::tuple<int, int, int, int, int> parseCmdLineArgs(int argc, char *argv[]) {
    int M = 1024;
    int N = 1024;
    int K = 1024;
    int num_trials = 10;
    int num_warmups = 10;

    for (int i = 1; i < argc; i++) {
        std::string option(argv[i]);
        if (i + 1 < argc) {
            std::string value(argv[i + 1]);
            if (option == "-m") {
                M = std::stoi(value);
                i++;
            } else if (option == "-n") {
                N = std::stoi(value);
                i++;
            } else if (option == "-k") {
                K = std::stoi(value);
                i++;
            } else if (option == "--num_trials") {
                num_trials = std::stoi(value);
                i++;
            } else if (option == "--num_warmups") {
                num_warmups = std::stoi(value);
                i++;
            }
        } else {
            std::cerr << "Missing value for option " << option << std::endl;
        }
    }

    return std::make_tuple(M, N, K, num_trials, num_warmups);
}

int main(int argc, char *argv[]) {
    std::tuple<int, int, int, int, int> parsedCmdLineArgsTuple = parseCmdLineArgs(argc, argv);
    int M = std::get<0>(parsedCmdLineArgsTuple);
    int N = std::get<1>(parsedCmdLineArgsTuple);
    int K = std::get<2>(parsedCmdLineArgsTuple);
    int numTrials = std::get<3>(parsedCmdLineArgsTuple);
    int nWarmUps = std::get<4>(parsedCmdLineArgsTuple);

    int* h_a = (int*)malloc(M * K * sizeof(int));
    int* h_b = (int*)malloc(K * N * sizeof(int));
    int* h_c = (int*)malloc(M * N * sizeof(int));

    srand(42);

    for (size_t i = 0; i < M; i++) {
        for (size_t j = 0; j < K; j++) {
            h_a[i * K + j] = rand() % 10;
        }
    }

    for (size_t i = 0; i < K; i++) {
        for (size_t j = 0; j < N; j++) {
            h_b[i * N + j] = rand() % 10;
        }
    }

    int *d_a, *d_b, *d_c;
    cudaMalloc((int**)&d_a, M * K * sizeof(int));
    cudaMalloc((int**)&d_b, K * N * sizeof(int));
    cudaMalloc((int**)&d_c, M * N * sizeof(int));

    copyFromHostToDevice<int>(h_a, h_b, d_a, d_b, M, N, K);

    for (int i = 0; i < nWarmUps; i++) {
        executeKernel<int>(d_a, d_b, d_c, M, N, K);
    }

    auto kernelLauncher = [&]() {
        executeKernel<int>(d_a, d_b, d_c, M, N, K);
    };

    benchmarkKernel(kernelLauncher, numTrials);

    copyFromDeviceToHost<int>(d_c, h_c, M, N);
    //verifyResult<int>(h_a, h_b, h_c, M, N, K);
    deallocateMemory<int>(d_a, d_b, d_c);
    cleanUpDevice();

    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}

Overwriting gemm_naive_2stream.cu


In [ ]:
# @title
%%writefile gemm_naive_2stream_masking.cu
#include <iostream>
#include <tuple>
#include <cassert>
#include "libsmctrl/libsmctrl.h"

using namespace std;

template<typename T>
__global__ void matmul_kernel(T const* a, T const* b, T* c, int M, int N, int K, int row_offset) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    int global_row = row + row_offset;

    if (col < N && row < M) {
        T sum = 0;
        for (int k = 0; k < K; k++) {
            sum += a[global_row * K + k] * b[k * N + col];
        }
        c[global_row * N + col] = sum;
    }
}

template<typename T>
__host__ void verifyResult(T* a, T* b, T* c, int M, int N, int K) {
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < N; j++) {
            T sum = 0;
            for (int k = 0; k < K; k++) {
                sum += a[i * K + k] * b[k * N + j];
            }
            assert(c[i * N + j] == sum);
        }
    }
    cout << "Result is correct";
}

template<typename T>
__host__ void copyFromHostToDevice(T* h_a, T* h_b, T* d_a, T* d_b, int M, int N, int K) {
    size_t a_bytes = M * K * sizeof(T);
    size_t b_bytes = K * N * sizeof(T);
    cudaError_t err = cudaMemcpy(d_a, h_a, a_bytes, cudaMemcpyHostToDevice);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy h_a to d_a: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaMemcpy(d_b, h_b, b_bytes, cudaMemcpyHostToDevice);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy h_b to d_b: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template<typename T>
__host__ void executeKernel(T* d_a, T* d_b, T* d_c, int M, int N, int K) {
    int nStreams = 2;
    uint64_t streamMasks[2] = {0xFFFFFFFFFFFFFC00, 0xFFFFFFFFFFF003FF};

    cudaStream_t stream[nStreams];
    for (int i = 0; i < nStreams; i ++) {
        cudaStreamCreate(&stream[i]);
        libsmctrl_set_stream_mask(&stream[i], streamMasks[i]);
    }

    int BLOCK_DIM = 32;
    dim3 block(BLOCK_DIM, BLOCK_DIM, 1);

    int M1 = M / 2;          // first half
    int M2 = M - M1;         // remaining rows

    dim3 grid1((N + BLOCK_DIM - 1) / BLOCK_DIM, (M1 + BLOCK_DIM - 1) / BLOCK_DIM, 1);
    dim3 grid2((N + BLOCK_DIM - 1) / BLOCK_DIM, (M2 + BLOCK_DIM - 1) / BLOCK_DIM, 1);

    matmul_kernel<T><<<grid1, block, 0, stream[0]>>>(d_a, d_b, d_c, M1, N, K, 0);
    matmul_kernel<T><<<grid2, block, 0, stream[1]>>>(d_a, d_b, d_c, M2, N, K, M1);

    for (int i = 0; i < nStreams; i ++) {
        cudaStreamSynchronize(stream[i]);
    }

    for (int i = 0; i < nStreams; i ++) {
        cudaStreamDestroy(stream[i]);
    }

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to launch kernel: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template <typename KernelFunc>
void benchmarkKernel(KernelFunc kernelFunc, int numTrials) {
    float* timings = new float[numTrials];

    // Create CUDA events for timing.
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Run the provided kernel launcher function multiple times.
    for (int i = 0; i < numTrials; i++) {
        cudaEventRecord(start, 0);
        kernelFunc();  // Launch the kernel via the callable.
        cudaEventRecord(stop, 0);
        cudaEventSynchronize(stop);

        float ms;
        cudaEventElapsedTime(&ms, start, stop);
        timings[i] = ms;
    }

    // Clean up CUDA events.
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    // Compute the average execution time.
    float sum = 0.0f;
    for (int i = 0; i < numTrials; i++) {
        sum += timings[i];
    }
    float average = sum / numTrials;

    // Compute the standard deviation.
    float variance = 0.0f;
    for (int i = 0; i < numTrials; i++) {
        float diff = timings[i] - average;
        variance += diff * diff;
    }
    variance /= numTrials;
    float stddev = sqrt(variance);

    // Print the results.
    printf("-----NUM TRIALS: %d -----\n", numTrials);
    printf("Average execution time: %f ms\n", average);
    printf("Standard deviation: %f ms\n", stddev);

    delete[] timings;
}

template<typename T>
__host__ void copyFromDeviceToHost(T* d_c, T* h_c, int M, int N) {
    size_t bytes = M * N * sizeof(T);
    cudaError_t err = cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to copy d_c to h_c: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

template<typename T>
__host__ void deallocateMemory(T* d_a, T* d_b, T* d_c) {
    cudaError_t err = cudaFree(d_a);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_a: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaFree(d_b);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_b: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
    err = cudaFree(d_c);
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to free d_c: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

__host__ void cleanUpDevice() {
    cudaError_t err = cudaDeviceReset();
    if (err != cudaSuccess) {
        fprintf(stderr, "Failed to clean up device: error code %s", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }
}

__host__ std::tuple<int, int, int, int, int> parseCmdLineArgs(int argc, char *argv[]) {
    int M = 1024;
    int N = 1024;
    int K = 1024;
    int num_trials = 10;
    int num_warmups = 10;

    for (int i = 1; i < argc; i++) {
        std::string option(argv[i]);
        if (i + 1 < argc) {
            std::string value(argv[i + 1]);
            if (option == "-m") {
                M = std::stoi(value);
                i++;
            } else if (option == "-n") {
                N = std::stoi(value);
                i++;
            } else if (option == "-k") {
                K = std::stoi(value);
                i++;
            } else if (option == "--num_trials") {
                num_trials = std::stoi(value);
                i++;
            } else if (option == "--num_warmups") {
                num_warmups = std::stoi(value);
                i++;
            }
        } else {
            std::cerr << "Missing value for option " << option << std::endl;
        }
    }

    return std::make_tuple(M, N, K, num_trials, num_warmups);
}

int main(int argc, char *argv[]) {
    std::tuple<int, int, int, int, int> parsedCmdLineArgsTuple = parseCmdLineArgs(argc, argv);
    int M = std::get<0>(parsedCmdLineArgsTuple);
    int N = std::get<1>(parsedCmdLineArgsTuple);
    int K = std::get<2>(parsedCmdLineArgsTuple);
    int numTrials = std::get<3>(parsedCmdLineArgsTuple);
    int nWarmUps = std::get<4>(parsedCmdLineArgsTuple);

    int* h_a = (int*)malloc(M * K * sizeof(int));
    int* h_b = (int*)malloc(K * N * sizeof(int));
    int* h_c = (int*)malloc(M * N * sizeof(int));

    srand(42);

    for (size_t i = 0; i < M; i++) {
        for (size_t j = 0; j < K; j++) {
            h_a[i * K + j] = rand() % 10;
        }
    }

    for (size_t i = 0; i < K; i++) {
        for (size_t j = 0; j < N; j++) {
            h_b[i * N + j] = rand() % 10;
        }
    }

    int *d_a, *d_b, *d_c;
    cudaMalloc((int**)&d_a, M * K * sizeof(int));
    cudaMalloc((int**)&d_b, K * N * sizeof(int));
    cudaMalloc((int**)&d_c, M * N * sizeof(int));

    copyFromHostToDevice<int>(h_a, h_b, d_a, d_b, M, N, K);

    for (int i = 0; i < nWarmUps; i++) {
        executeKernel<int>(d_a, d_b, d_c, M, N, K);
    }

    auto kernelLauncher = [&]() {
        executeKernel<int>(d_a, d_b, d_c, M, N, K);
    };

    benchmarkKernel(kernelLauncher, numTrials);

    copyFromDeviceToHost<int>(d_c, h_c, M, N);
    //verifyResult<int>(h_a, h_b, h_c, M, N, K);
    deallocateMemory<int>(d_a, d_b, d_c);
    cleanUpDevice();

    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}

Overwriting gemm_naive_2stream_masking.cu


In [8]:
!nvcc -I./libsmctrl -L./libsmctrl -lsmctrl -lcuda -o gemm_naive_2stream_masking gemm_naive_2stream_masking.cu -arch=sm_75
!./gemm_naive_2stream_masking -m 3840 -n 3840 -k 3840 --num_trials 1000 --num_warmups 100

-----NUM TRIALS: 1000 -----
Average execution time: 187.880203 ms
Standard deviation: 0.441321 ms


In [9]:
!nvcc -o gemm_naive_2stream gemm_naive_2stream.cu -arch=sm_75
!./gemm_naive_2stream -m 3840 -n 3840 -k 3840 --num_trials 1000 --num_warmups 100

-----NUM TRIALS: 1000 -----
Average execution time: 186.428772 ms
Standard deviation: 0.942849 ms
